In [0]:
%sql
-- DROP SCHEMA IF EXISTS data_dev_olist.gold CASCADE;


In [0]:
# Databricks Notebook
# MAGIC %md # Gold Layer - Star Schema (Customer Dimension Only)

# COMMAND ----------
# IMPORT CÁC THƯ VIỆN CẦN THIẾT
from pyspark.sql.functions import *
from pyspark.sql.types import *

# CONFIG CONFIGURATION
CATALOG_NAME = "data_dev_olist"
SCHEMA_NAME = "gold"

# Thiết lập sử dụng Catalog gốc
spark.sql(f"USE CATALOG {CATALOG_NAME}")

# Tạo schema gold nếu chưa tồn tại (Vẫn để mặc định vì ta sẽ override vị trí lưu trữ ở mức bảng)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}")

# COMMAND ----------
# DBTITLE 1, HELPER FUNCTION: SAVE GOLD TABLE
def save_gold_table(df, table_name):
    # Định nghĩa chính xác đường dẫn vật lý trên ADLS của bạn
    gold_path = f"abfss://raw-data@quocluudata.dfs.core.windows.net/gold/{table_name}"
    target_table = f"{CATALOG_NAME}.{SCHEMA_NAME}.{table_name}"

    print(f"🚀 Processing: {table_name}")

    # Kiểm tra thư mục vật lý tồn tại trên ADLS chưa
    is_path_exists = False
    try:
        dbutils.fs.ls(gold_path)
        is_path_exists = True
    except:
        is_path_exists = False

    if not is_path_exists:
        print(f"✨ Creating table: {target_table}")
        (
            df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .option("path", gold_path)     # Ép buộc lưu tại ADLS của quocluudata
            .saveAsTable(target_table)     # Đăng ký External Table vào Unity Catalog
        )
    else:
        print(f"🔄 Refreshing table: {target_table}")
        (
            df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .option("path", gold_path)     # Ghi đè vào thư mục cũ, giữ cấu trúc External Table
            .saveAsTable(target_table)
        )

    print(f"✅ Completed: {target_table}")
    
    # Đọc ngược lại từ catalog để kiểm tra số dòng thực tế
    final_count = spark.table(target_table).count()
    print(f"📊 Rows: {final_count}")
    print("-" * 50)

# COMMAND ----------
# DBTITLE 1, PROCESS: dim_customer
# 1. Đọc dữ liệu sạch từ tầng Silver
silver_cleaned_customer = spark.table("silver.clean_customer")
silver_cleaned_geolocation = spark.table("silver.clean_geolocation")

# 2. Xử lý làm sạch tọa độ (loại bỏ trùng lặp zip code để chống nhân dòng)
geo_unique = silver_cleaned_geolocation.dropDuplicates(["geolocation_zip_code_prefix"])

# 3. Thực hiện phép nối Left Join
joined_customer = silver_cleaned_customer.join(
    geo_unique,
    col("customer_zip_code_prefix") == col("geolocation_zip_code_prefix"),
    how="left"
)

# 4. Chuẩn hóa tên cột và loại bỏ cột thừa của tầng chứa
dim_customer_df = (joined_customer
    .withColumnRenamed("geolocation_lat", "customer_lat")
    .withColumnRenamed("geolocation_lng", "customer_lng")
    .withColumn("customer_city", coalesce(col("geolocation_city"), col("customer_city")))
    .withColumn("customer_state", coalesce(col("geolocation_state"), col("customer_state")))
    .select(
        "customer_id", 
        "customer_unique_id", 
        "customer_city", 
        "customer_state", 
        "customer_lat", 
        "customer_lng"
    )
    .dropDuplicates(["customer_id"])
)

# 5. Kích hoạt hàm Helper để lưu dữ liệu an toàn
save_gold_table(dim_customer_df, "dim_customer")

# COMMAND ----------
# DBTITLE 1, dim_seller
silver_cleaned_seller = spark.table("silver.clean_seller")

dim_seller_df = silver_cleaned_seller.select(
    "seller_id", 
    "seller_zip_code_prefix", 
    "seller_city", 
    "seller_state"
).dropDuplicates(["seller_id"])

save_gold_table(dim_seller_df, "dim_seller")

# COMMAND ----------
# DBTITLE 1, dim_review
silver_cleaned_order_review = spark.table("silver.clean_order_review")

dim_review_df = silver_cleaned_order_review.select(
    "review_id", 
    "review_score"
).dropDuplicates(["review_id"])

save_gold_table(dim_review_df, "dim_review")

# COMMAND ----------
# DBTITLE 1, dim_product
silver_cleaned_product = spark.table("silver.clean_product")
silver_cleaned_product_category = spark.table("silver.clean_product_category")

# Xử lý động cho lỗi chính tả (lenght -> length) của Olist
prod_cols = silver_cleaned_product.columns
name_col = "product_name_length" if "product_name_length" in prod_cols else "product_name_lenght"
desc_col = "product_description_length" if "product_description_length" in prod_cols else "product_description_lenght"

dim_product_df = silver_cleaned_product.join(
    silver_cleaned_product_category,
    "product_category_name",
    "inner"
).select(
    "product_id", 
    "product_category_name", 
    "product_category_name_english",
    col(name_col).alias("product_name_length"),
    col(desc_col).alias("product_description_length"),
    "product_photos_qty", 
    "product_weight_g", 
    "product_length_cm", 
    "product_height_cm", 
    "product_width_cm"
).dropDuplicates(["product_id"])

save_gold_table(dim_product_df, "dim_product")

# COMMAND ----------
# DBTITLE 1, dim_order
silver_cleaned_order = spark.table("silver.clean_order")

dim_order_df = silver_cleaned_order.select(
    "order_id", 
    "order_status"
).dropDuplicates(["order_id"])

save_gold_table(dim_order_df, "dim_order")

# COMMAND ----------
# DBTITLE 1, dim_date    
silver_clean_order = spark.table("silver.clean_order")

start_date_val = silver_clean_order.select(min("order_purchase_timestamp")).first()[0]
end_date_val = silver_clean_order.select(max("order_purchase_timestamp")).first()[0]

start_date = start_date_val.date() if start_date_val else None
end_date = end_date_val.date() if end_date_val else None

if start_date and end_date:
    days_count = (end_date - start_date).days + 1
    
    date_df = spark.range(0, days_count).withColumn(
        "full_date", 
        lit(start_date) + col("id").cast("string").cast("interval day")
    )

    dim_date_df = (date_df
        .withColumn("dateKey", year(col("full_date")) * 10000 + month(col("full_date")) * 100 + dayofmonth(col("full_date")))
        .withColumn("year", year(col("full_date")))
        .withColumn("quarter", quarter(col("full_date")))
        .withColumn("month", month(col("full_date")))
        .withColumn("week", weekofyear(col("full_date")))
        .withColumn("day", dayofmonth(col("full_date")))
        .withColumn("day_of_year", dayofyear(col("full_date")))
        .withColumn("day_name_of_week", date_format(col("full_date"), "EEEE"))
        .withColumn("month_name_of_week", date_format(col("full_date"), "MMMM"))
        .select(
            "dateKey", "full_date", "year", "quarter", "month", 
            "week", "day", "day_of_year", "day_name_of_week", "month_name_of_week"
        )
    )
    save_gold_table(dim_date_df, "dim_date")
else:
    raise ValueError("Could not calculate min/max dates from silver.clean_order")

# COMMAND ----------
# MAGIC %md ## 3. Bảng Fact (Fact Table)

# COMMAND ----------
# DBTITLE 1, fact_table
silver_cleaned_order_item = spark.table("silver.clean_order_item")
silver_cleaned_order = spark.table("silver.clean_order")
silver_cleaned_payment = spark.table("silver.clean_payment")
silver_cleaned_order_review = spark.table("silver.clean_order_review")

dim_customer = spark.table("gold.dim_customer")
dim_seller = spark.table("gold.dim_seller")
dim_product = spark.table("gold.dim_product")
dim_order = spark.table("gold.dim_order")
dim_date = spark.table("gold.dim_date")

# CHỐNG NHÂN DÒNG 1: Gom nhóm (Aggregate) bảng thanh toán theo từng đơn hàng
payment_agg = silver_cleaned_payment.groupBy("order_id").agg(
    sum("payment_value").alias("total_payment_value"),
    max("payment_installments").alias("max_payment_installments"),
    max("payment_sequential").alias("max_payment_sequential")
)

# CHỐNG NHÂN DÒNG 2: Loại bỏ review trùng lặp (1 order_id chỉ lấy 1 review_id duy nhất)
review_dedup = silver_cleaned_order_review.dropDuplicates(["order_id"])

# Hạt dữ liệu cơ sở: Kết hợp Order và Order Item
base_df = silver_cleaned_order.join(silver_cleaned_order_item, "order_id", "inner")

# Thực hiện kiến trúc nối hình sao (Star Schema)
fact_table_df = (base_df
    .join(dim_order, on="order_id", how="inner")
    .join(dim_product, on="product_id", how="inner")
    .join(dim_customer, on="customer_id", how="inner")
    .join(dim_seller, on="seller_id", how="inner")
    
    # Left join sang bảng payment đã được tính tổng tổng giá trị (Tránh nhân dòng chi phí)
    .join(payment_agg, on="order_id", how="left")
    
    # Left join sang bảng review đã lọc trùng
    .join(review_dedup, on="order_id", how="left")
    
    # Join tối ưu với Dim Date (Chuyển timestamp thành Date thô để trùng khớp kiểu dữ liệu)
    .join(
        dim_date,
        to_date(col("order_purchase_timestamp")) == col("full_date"),
        how="inner"
    )
    .select(
        "order_id",
        # Sử dụng pk_hash của tầng silver làm id chi tiết của sản phẩm trong đơn hàng nếu có
        col("pk_hash").alias("order_item_unique_id") if "pk_hash" in base_df.columns else "order_item_id",
        "customer_id", 
        "product_id", 
        coalesce(col("review_id"), lit("N/A")).alias("review_id"),
        "seller_id", 
        "dateKey", 
        "price", 
        "freight_value",
        col("total_payment_value").alias("payment_value"),
        col("max_payment_installments").alias("payment_installments"),
        col("max_payment_sequential").alias("payment_sequential")
    )
)

# Ghi và tạo bảng Fact bằng hàm Helper an toàn
save_gold_table(fact_table_df, "fact_table")

print("🎉 HOÀN THÀNH XÂY DỰNG MÔ HÌNH STAR SCHEMA TẦNG GOLD!")

🚀 Processing: dim_customer
🔄 Refreshing table: data_dev_olist.gold.dim_customer
✅ Completed: data_dev_olist.gold.dim_customer
📊 Rows: 99441
--------------------------------------------------
🚀 Processing: dim_seller
🔄 Refreshing table: data_dev_olist.gold.dim_seller
✅ Completed: data_dev_olist.gold.dim_seller
📊 Rows: 3095
--------------------------------------------------
🚀 Processing: dim_review
🔄 Refreshing table: data_dev_olist.gold.dim_review
✅ Completed: data_dev_olist.gold.dim_review
📊 Rows: 8884
--------------------------------------------------
🚀 Processing: dim_product
🔄 Refreshing table: data_dev_olist.gold.dim_product
✅ Completed: data_dev_olist.gold.dim_product
📊 Rows: 32951
--------------------------------------------------
🚀 Processing: dim_order
🔄 Refreshing table: data_dev_olist.gold.dim_order
✅ Completed: data_dev_olist.gold.dim_order
📊 Rows: 99441
--------------------------------------------------
🚀 Processing: dim_date
🔄 Refreshing table: data_dev_olist.gold.dim_date

In [0]:
print(dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get())

/Users/denoiadotterer91@hotmail.com/data olist/Bronze Delta/gold 


In [0]:
%sql
DESCRIBE EXTENDED data_dev_olist.gold.dim_customer;


col_name,data_type,comment
customer_id,string,null
customer_unique_id,string,null
customer_city,string,null
customer_state,string,null
customer_lat,double,null
customer_lng,double,null
,,
# Delta Statistics Columns,,
Column Names,"customer_id, customer_lng, customer_state, customer_unique_id, customer_city, customer_lat",
Column Selection Method,first-32,


In [0]:
%sql
-- DESCRIBE EXTENDED data_dev_olist.silver.clean_customer;


In [0]:
%sql 
-- DESCRIBE SCHEMA EXTENDED data_dev_olist.gold;

In [0]:
%sql 
DESCRIBE CATALOG EXTENDED data_dev_olist;

info_name,info_value
Catalog Name,data_dev_olist
Comment,
Owner,denoiadotterer91@hotmail.com
Catalog Type,Regular
Created By,denoiadotterer91@hotmail.com
Created At,2026-06-10 AD at 04:51:48 UTC
Updated By,denoiadotterer91@hotmail.com
Updated At,2026-06-11 AD at 16:34:51 UTC
Storage Root,abfss://quocluudata@markmiller89bamboomedia.dfs.core.windows.net/
Storage Location,abfss://quocluudata@markmiller89bamboomedia.dfs.core.windows.net/__unitystorage/catalogs/862deb96-5562-4641-9be8-2b6f4c66ead1


In [0]:
%sql
SHOW EXTERNAL LOCATIONS;

name,url,comment
data_dev,abfss://raw-data@quocluudata.dfs.core.windows.net/,
ext_tabke,abfss://quocluudata@markmiller89bamboomedia.dfs.core.windows.net/,null
quocluu,abfss://unity-catalog-storage@dbstoragew4tev4ewrda7o.dfs.core.windows.net/7405612683717837,null


In [0]:
%sql
 table data_dev_olist.silver.clean_customer;

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,is_active,_processed_at
237098a64674ae89babdc426746260fc,4390ddbb6276a66ff1736a6710205dca,82820,curitiba,PR,true,2026-06-19T05:00:40.819Z
e3109970a3fe8021d5ff82c577ce5606,a8654e2af5da6bb72f52c22b164855e1,5528,sao paulo,SP,true,2026-06-19T05:00:40.819Z
c532a74a3ebf1bacce2e2bcce3783317,91ec50a00ae74d0a229d2efdf4344e1e,14026,ribeirao preto,SP,true,2026-06-19T05:00:40.819Z
19cecb194f54e614b70d971306a9931b,d251c190ca75786e9ab937982d60d1d4,30320,belo horizonte,MG,true,2026-06-19T05:00:40.819Z
c82a5e4fafdbeb34f08928ccfba27d14,ca19a17e381182923b66007a351574b7,85854,foz do iguacu,PR,true,2026-06-19T05:00:40.819Z
b06429ef920fcfdd75713c712c9ee7b7,9316f45a5da8403a5938bd6069b1a4a7,12240,sao jose dos campos,SP,true,2026-06-19T05:00:40.819Z
d3ab15f0bd2c58865d566ab645572cd5,9ccfff93c79f3dd996cce15f26480c5b,21615,rio de janeiro,RJ,true,2026-06-19T05:00:40.819Z
031cd5f826be3d804771e3e3a1b21a1c,717aa48025662fcf27ddebbecc5f782b,41706,salvador,BA,true,2026-06-19T05:00:40.819Z
79adcf02229a33e78f0f5412a2434f53,8c8fccc50566baaed602e3775d9d5665,21515,rio de janeiro,RJ,true,2026-06-19T05:00:40.819Z
e3c7e245a96d7fa339fe6c16f8da4e90,79051ee5ee98c4bd6982e67e2e79dbcb,7847,franco da rocha,SP,true,2026-06-19T05:00:40.819Z


In [0]:
%sql
-- DESCRIBE EXTENDED data_dev_olist.silver.clean_customer;


In [0]:
%sql 
SELECT 
    COUNT(order_id) AS total_rows_silver,                -- Tổng số dòng
    COUNT(DISTINCT order_id) AS unique_orders_silver    -- Tổng số đơn thực tế (Con số chuẩn)
FROM data_dev_olist.silver.clean_order;


total_rows_silver,unique_orders_silver
99441,99441


In [0]:
%sql
SELECT 
    COUNT(*) AS total_rows_gold,                      -- Tổng số dòng trong bảng Fact
    COUNT(DISTINCT order_id) AS unique_orders_gold    -- Số đơn độc nhất trong bảng Fact
FROM data_dev_olist.gold.fact_table;

total_rows_gold,unique_orders_gold
112650,98666


In [0]:
%sql
SELECT 
    d.order_status,
    COUNT(DISTINCT f.order_id) AS so_don_thuc_te
FROM data_dev_olist.gold.fact_table f
JOIN data_dev_olist.gold.dim_order d ON f.order_id = d.order_id
GROUP BY d.order_status;

order_status,so_don_thuc_te
invoiced,312
processing,301
shipped,1106
unavailable,6
delivered,96478
canceled,461
approved,2
